# Benchmark-relative Brent and WTI forecasting — corrected IRFA workflow

This notebook is the executable companion to the manuscript. It applies market-appropriate targets:

- **Brent:** next-common-observation log return.
- **WTI:** next-common-observation dollar change (USD/bbl). The negative WTI spot price of **−36.98 USD/bbl on 20 April 2020 is retained**; no positive-price filter or logarithm is applied to WTI.

The lagged-only cross-market sensitivity removes only the contemporaneous other-market movement, current spread and unlagged other-market rolling-volatility features; it retains lagged other-market movements, lagged volatility summaries (`other_vol_lag1_*`) and lagged spread terms. The optional REFIT selector mirrors this feature rule.

The default **AUDIT** mode uses the frozen paired forecast panel and independently recalculates the reported metrics, DM/HAC/HLN tests with Holm adjustments, AR(5)-only Clark–West diagnostic, event-window metrics, delayed one-barrel P&L including terminal liquidation, and block-bootstrap intervals. It checks those recalculations against the canonical supplementary CSVs at 1e-10 absolute tolerance.

The optional **REFIT** mode performs a fresh annual expanding-window estimation using the documented feature recipe and selected settings. Original fitted model objects and the complete historical software environment were not preserved; REFIT is not a bitwise replay. A current-environment WTI XGBoost terminal-window refit did not reproduce the archived prediction segment. Do not replace the frozen estimates until a complete new run is validated and all manuscript outputs are regenerated.

**SHAP is not reported in the manuscript or submission figures.** Archived SHAP metadata conflicted on whether the analysis used a quarterly or annual window, and WTI attribution could not be independently reconciled to the frozen model predictions. An optional exploratory attribution may be generated by REFIT, but it is not a reported result.

In [ ]:
# Colab setup: upload the corrected package ZIP and extract it.
from google.colab import files
from pathlib import Path
import zipfile, os, glob, sys, subprocess
uploaded = files.upload()
zip_candidates = [name for name in uploaded if name.lower().endswith('.zip')]
if not zip_candidates:
    raise ValueError('Please upload the corrected IRFA submission package ZIP.')
archive = zip_candidates[0]
ROOT = Path('/content/irfa_package')
ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive) as zf:
    bad = zf.testzip()
    if bad:
        raise RuntimeError(f'Corrupt ZIP member: {bad}')
    zf.extractall(ROOT)
# Find the package folder independent of its version suffix.
package_roots = sorted([p for p in ROOT.glob('IRFA_Oil_Forecasting_Package_*') if p.is_dir()])
package_root = package_roots[0] if package_roots else ROOT
requirements = package_root / 'Colab' / 'requirements-colab.txt'
if requirements.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)
script = package_root / 'Colab' / 'IRFA_Oil_Spot_Forecasting_Corrected.py'
if not script.exists():
    raise FileNotFoundError(f'Could not find forecasting script under {package_root}')
os.environ['IRFA_ROOT'] = str(package_root)
print('Package root:', package_root)
print('Python script:', script)
print('Expected package audit data:', package_root / 'Supplementary' / 'Corrected_Run_2026-10-30')

## 1. Verify the archived results (recommended)

The default AUDIT run fails closed if the frozen files differ from the manuscript tables. It checks the 8,425 matched prices, the negative WTI observation, the 8,424 one-step movements, the 61-row feature warm-up, the 1,068 initial training pairs, and identical 7,295-target date sets. It recalculates market/model metrics, all market × specification × loss DM/HAC/HLN tests and Holm adjustments, the AR(5)-only Clark–West check, event-window metrics, delayed one-barrel P&L with initial entry/rebalancing/terminal liquidation, and both block-bootstrap tables. The script writes reconciliation CSVs and a machine-readable audit summary. SHAP is excluded from the reported result set.

In [ ]:
# AUDIT is the default and reproduces the reported summary tables from frozen forecasts.
import subprocess, sys
os.environ['IRFA_MODE'] = 'AUDIT'
run = subprocess.run([sys.executable, str(script)], env=os.environ.copy(), text=True, capture_output=True)
print(run.stdout)
if run.returncode != 0:
    print(run.stderr)
    raise RuntimeError('Audit failed')
print('Audit output:', package_root / 'Colab_Reproduction_Output')

In [ ]:
# Inspect reconciled metric differences; values should be at floating-point tolerance.
import pandas as pd
out = package_root / 'Colab_Reproduction_Output'
recon = pd.read_csv(out / 'metric_reconciliation.csv')
display(recon[['market','model','specification','rmse_abs_diff','mae_abs_diff']].head(20))
print('Maximum absolute RMSE difference:', recon.rmse_abs_diff.max())
print('Maximum absolute MAE difference:', recon.mae_abs_diff.max())
dm = pd.read_csv(out / 'DM_HAC_tests_recalculated.csv')
display(dm[['market','specification','model','loss','mean_diff','hac_stat','p_value','p_holm']])

## 2. Optional fresh model re-estimation

To fit the models again, edit the script configuration near the top and set `MODE = "REFIT"`. Leave `REFRESH_LIVE_DATA = False` to use the archived aligned spot-price snapshot, or set it to `True` to download current EIA-origin/DataHub CSV files.

The refit uses annual expanding windows, an initial minimum of 1,000 training observations and the selected hyperparameters recorded with the empirical run. It creates a new forecast panel and recomputes metrics, HAC/HLN + Holm tests, AR(5)-only Clark–West, event diagnostics, delayed P&L (including terminal exit), and block-bootstrap intervals. It is a fresh estimation, not a bitwise replay: original fitted estimator objects and the complete historical software environment were not preserved, and a current-environment WTI XGBoost terminal-window re-fit did not reproduce the archived prediction segment. Do not replace the paper outputs until a full new run has completed and every output and manuscript number has been validated. SHAP attribution is not generated or reported in this release.

The REFIT sensitivity feature selector matches the frozen archived rule: remove `other_move_0`, `spread_0` and unlagged `other_vol_*`; retain `other_move_lag*`, `other_vol_lag1_*` and `spread_lag1`, `spread_lag2`, `spread_lag5`. A REFIT is still a fresh estimation and is not guaranteed to reproduce the archived model-level forecast values.

In [ ]:
# Optional: set the source file's MODE = 'REFIT' before running this cell.
# This cell is intentionally not run by default.
# os.environ['IRFA_MODE'] = 'REFIT'
# run = subprocess.run([sys.executable, str(script)], env=os.environ.copy(), text=True, capture_output=True)
# print(run.stdout)
# if run.returncode != 0:
#     print(run.stderr)
#     raise RuntimeError('Refit failed')

## 3. Interpretation and release safeguards

1. Negative WTI spot prices must never be deleted or passed into a logarithm.
2. Brent RMSE/MAE are reported in log-return basis points; WTI RMSE/MAE are reported in USD/bbl. Do not pool these target units.
3. The zero-change random walk has an identically zero forecast movement; directional accuracy and directional N are **N/A**.
4. In the DM loss differential, `candidate loss − benchmark loss`, negative values favour the candidate.
5. Holm families are separate by market, predictor specification and loss type. Event and P&L analyses are exploratory.
6. The delayed P&L proxy includes initial entry, rebalancing and terminal liquidation, but it is not an executable trading backtest.
7. The contemporaneous cross-market specification is not strict real-time evidence without historical publication timestamps.
8. AUDIT reproduces reported statistics from frozen predictions; REFIT is a new estimation and may differ. The WTI XGBoost refit discrepancy must be resolved before claiming full model-level reproducibility.
9. SHAP is not included in the paper because its archived window metadata and WTI model attribution could not be reconciled.

8. Supplementary Table S1 reports the exact price-to-forecast sample waterfall. Supplementary Table S2 reports the exact event-window dates and annual training cutoffs; the 2023 post-anchor window begins on 10 October because 9 October is not in the common target-date panel.
9. The delayed P&L Sharpe is computed from daily net USD/bbl P&L, and drawdown is an additive USD/bbl peak-to-trough quantity; neither is a compounded percentage return.